#### This notebook visualizes AORC data and develops eight baselines for the temporal downscaling of the rainfall
##### Author: Omid Emamjomehzadeh (https://www.omidemam.com/)
##### Supervisor: Dr. Omar Wani (https://engineering.nyu.edu/faculty/omar-wani)
##### Hydrologic Systems Group @NYU (https://www.omarwani.com/)

In [1]:
import os
import torch
from model import Generator  
from train import train
#from train_profile import train
from model import Generator, Discriminator
import torch.optim as optim
from data import readFile, make_temporal_batches
import xarray as xr
import numpy as np
from einops import rearrange
import datetime
import matplotlib.pyplot as plt
import matplotlib
import argparse
import pandas as pd

# load Daily data

In [2]:
def get_data(year):

    input = readFile(f'/scratch/jl14811/GCM_1981-2011/CESM2/pr_{year}*.nc', 'pr', 1, 21, 31) * 3600
    input = np.concatenate(input, axis=0)
    mean_val = np.nanmean(input)
    input = np.where(np.isnan(input), 0.0, input)
    print(input.shape)
    return input

In [ ]:
def get_data(year):

    input = readFile(f'/scratch/jl14811/AORC_1981-2011/AORC_21_31/APCP_surface_{year}*.nc', 'APCP_surface', 24, 21, 31)
    input = input.sum(axis=1)
    mean_val = np.nanmean(input)
    input = np.where(np.isnan(input), 0.0, input)
    print(input.shape)
    return input

# Baseline 1
#### Conserve the mass with minimum assumption (constant function)



In [3]:
for year in range(1981, 2012):
    input = get_data(year)
    print(input.shape)
    # Output directory
    output_dir = "/scratch/jl14811/validation/Baselines/baseline1/"
    os.makedirs(output_dir, exist_ok=True)
    # Create 6-hourly flat maps
    days = input.shape[0]
    for day in range(days):
        for hour in range(4):
            baseline_hour = np.expand_dims(input[day] / 4, axis=0)
            out_path = os.path.join(output_dir, f"baseline1_{year}_{day+1}_{hour*6}_{(hour+1)*6}.nc")
            baseline_hour = xr.Dataset(
            {
            "pr": (("time", "lat", "lon"), baseline_hour) 
            },
            )
            baseline_hour.to_netcdf(out_path)
            #print(f"Saved: {out_path}")
    

final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(3

# Baseline 2 
## Using the temporal pattern of region one of the northeastern, the first quantile 10 %


In [4]:
csv_path = r"/scratch/jl14811/validation/Baselines/ne_1_24h_temporal.csv"
# Skip header lines and load 48 rows of data
skip_lines = 9
df = pd.read_csv(csv_path, skiprows=skip_lines, nrows=48)
# Extract column '0.9' and add a zero at the start
values = df['0.9'].astype(float).to_numpy()
values = np.insert(values, 0, 0.0)
# Optional: check result
print(values.shape)
print(values)  # preview first few values
# Create slices for the even-indexed values
v_even_start = values[0:-2:2]  # [v0, v2, v4, ...]
v_even_end   = values[2::2]    # [v2, v4, v6, ...]
# Compute the differences
temp_pater_1 = v_even_end - v_even_start
# Print results
print("Shape:", temp_pater_1.shape)
print("First 5 differences:", temp_pater_1[:5])  # preview first few values

(49,)
[  0.    16.63  32.26  46.23  58.26  68.3   76.43  82.85  87.8   91.52
  94.25  96.2   97.56  98.48  99.08  99.46  99.69  99.83  99.91  99.95
  99.97  99.98  99.99  99.99  99.99  99.99  99.99  99.99  99.99  99.99
  99.99 100.   100.   100.   100.   100.   100.   100.   100.   100.
 100.   100.   100.   100.   100.   100.   100.   100.   100.  ]
Shape: (24,)
First 5 differences: [32.26 26.   18.17 11.37  6.45]


### Apply it to generate hourly precipitation fields for a day

In [5]:
for year in range(1981, 2012):
    input = get_data(year)
    # Output directory
    output_dir = "/scratch/jl14811/validation/Baselines/baseline2/"
    os.makedirs(output_dir, exist_ok=True)
    # Create 6-hourly flat 
    days = input.shape[0]
    for day in range(days):
        for i, hour in enumerate(range(4)):
            weight = temp_pater_1[i] / 100
            baseline_hour = np.expand_dims(input[day] * weight, axis=0)
            out_path = os.path.join(output_dir, f"baseline2_{year}_{day+1}_{hour*6}_{(hour+1)*6}.nc")
            baseline_hour = xr.Dataset(
            {
            "pr": (("time", "lat", "lon"), baseline_hour) 
            },
            )
            baseline_hour.to_netcdf(out_path)
            #print(f"Saved: {out_path}")

final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape

# Baseline 3
## Using the temporal pattern of region one of the northeastern, the fourth quantile 90 %

In [6]:
csv_path = r"/scratch/jl14811/validation/Baselines/ne_1_24h_temporal.csv"

# Skip header lines and load 48 rows of data
skip_lines = 225
df = pd.read_csv(csv_path, skiprows=skip_lines, nrows=48)
# Extract column '0.9' and add a zero at the start
values = df['0.1'].astype(float).to_numpy()
values = np.insert(values, 0, 0.0)
# Optional: check result
print(values.shape)
print(values)  # preview first few values
# Create slices for the even-indexed values
v_even_start = values[0:-2:2]  # [v0, v2, v4, ...]
v_even_end   = values[2::2]    # [v2, v4, v6, ...]
# Compute the differences
temp_pater_2 = v_even_end - v_even_start
# Print results
print("Shape:", temp_pater_2.shape)
print("First 5 differences:", temp_pater_2[:5])  # preview first few values

(49,)
[  0.     0.29   0.69   1.18   1.69   2.18   2.63   3.08   3.54   4.06
   4.67   5.37   6.19   7.1    8.09   9.14  10.24  11.38  12.56  13.8
  15.11  16.51  18.03  19.69  21.49  23.46  25.58  27.85  30.24  32.76
  35.37  38.08  40.9   43.82  46.88  50.1   53.52  57.14  60.99  65.05
  69.27  73.61  77.97  82.27  86.4   90.3   93.91  97.18 100.  ]
Shape: (24,)
First 5 differences: [0.69 1.   0.94 0.91 1.13]


### Apply it to generate hourly precipitation fields for a day

In [7]:
for year in range(1981, 2012):
    input = get_data(year)
    # Output directory
    output_dir = "/scratch/jl14811/validation/Baselines/baseline3/"
    os.makedirs(output_dir, exist_ok=True)
    # Create 6-hourly flat 
    days = input.shape[0]
    for day in range(days):
        for i, hour in enumerate(range(4)):
            weight = temp_pater_2[i] / 100
            baseline_hour = np.expand_dims(input[day] * weight, axis=0)
            out_path = os.path.join(output_dir, f"baseline3_{year}_{day+1}_{hour*6}_{(hour+1)*6}.nc")
            baseline_hour = xr.Dataset(
            {
            "pr": (("time", "lat", "lon"), baseline_hour) 
            },
            )
            baseline_hour.to_netcdf(out_path)
            #print(f"Saved: {out_path}")

final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape

# Baseline 4
## Using the temporal pattern of region one of the northeastern, the first quantile 50 %

In [8]:
csv_path = r"/scratch/jl14811/validation/Baselines/ne_1_24h_temporal.csv"
# Skip header lines and load 48 rows of data
skip_lines = 9
df = pd.read_csv(csv_path, skiprows=skip_lines, nrows=48)
# Extract column '0.9' and add a zero at the start
values = df['0.5'].astype(float).to_numpy()
values = np.insert(values, 0, 0.0)
# Optional: check result
print(values.shape)
print(values)  # preview first few values
# Create slices for the even-indexed values
v_even_start = values[0:-2:2]  # [v0, v2, v4, ...]
v_even_end   = values[2::2]    # [v2, v4, v6, ...]
# Compute the differences
temp_pater_3= v_even_end - v_even_start
# Print results
print("Shape:", temp_pater_3.shape)
print("First 5 differences:", temp_pater_3[:5])  

(49,)
[  0.     2.94   7.26  12.37  17.86  23.49  29.09  34.57  39.85  44.88
  49.6   53.98  57.97  61.56  64.72  67.48  69.86  71.9   73.66  75.21
  76.61  77.94  79.25  80.59  81.98  83.44  84.95  86.5   88.05  89.56
  91.    92.33  93.53  94.59  95.51  96.3   96.98  97.57  98.1   98.57
  98.99  99.35  99.64  99.83  99.94  99.98  99.98  99.99 100.  ]
Shape: (24,)
First 5 differences: [ 7.26 10.6  11.23 10.76  9.75]


### Apply it to generate hourly precipitation fields for a day

In [9]:
for year in range(1981, 2012):
    input = get_data(year)
    # Output directory
    output_dir = "/scratch/jl14811/validation/Baselines/baseline4/"
    os.makedirs(output_dir, exist_ok=True)
    # Create 6-hourly flat
    days = input.shape[0]
    for day in range(days):
        for i, hour in enumerate(range(4)):
            weight = temp_pater_3[i] / 100
            baseline_hour = np.expand_dims(input[day] * weight, axis=0) 
            out_path = os.path.join(output_dir, f"baseline4_{year}_{day+1}_{hour*6}_{(hour+1)*6}.nc")
            baseline_hour = xr.Dataset(
            {
            "pr": (("time", "lat", "lon"), baseline_hour) 
            },
            )
            baseline_hour.to_netcdf(out_path)
            #print(f"Saved: {out_path}")

final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape

# Baseline 5
## Using the temporal pattern of region one of the northeastern, the second quantile 50 %

In [10]:
csv_path = r"/scratch/jl14811/validation/Baselines/ne_1_24h_temporal.csv"
# Skip header lines and load 48 rows of data
skip_lines = 63
df = pd.read_csv(csv_path, skiprows=skip_lines, nrows=48)
# Extract column '0.9' and add a zero at the start
values = df['0.5'].astype(float).to_numpy()
values = np.insert(values, 0, 0.0)
# Optional: check result
print(values.shape)
print(values)  # preview first few values
# Create slices for the even-indexed values
v_even_start = values[0:-2:2]  # [v0, v2, v4, ...]
v_even_end   = values[2::2]    # [v2, v4, v6, ...]
# Compute the differences
temp_pater_4= v_even_end - v_even_start
# Print results
print("Shape:", temp_pater_4.shape)
print("First 5 differences:", temp_pater_4[:5])  

(49,)
[  0.     1.47   2.44   3.48   4.74   6.19   7.78   9.48  11.29  13.28
  15.51  18.07  21.02  24.39  28.19  32.38  36.91  41.69  46.63  51.61
  56.54  61.33  65.88  70.15  74.08  77.64  80.83  83.64  86.09  88.21
  90.03  91.58  92.9   94.03  95.    95.85  96.61  97.29  97.9   98.45
  98.93  99.33  99.63  99.83  99.93  99.96  99.96  99.98 100.  ]
Shape: (24,)
First 5 differences: [2.44 2.3  3.04 3.51 4.22]


### Apply it to generate hourly precipitation fields for a day

In [11]:
for year in range(1981, 2012):
    input = get_data(year)
    # Output directory
    output_dir = "/scratch/jl14811/validation/Baselines/baseline5/"
    os.makedirs(output_dir, exist_ok=True)
    # Create 6-hourly flat 
    days = input.shape[0]
    for day in range(days):
        for i, hour in enumerate(range(4)):
            weight = temp_pater_4[i] / 100
            baseline_hour = np.expand_dims(input[day] * weight, axis=0)
            out_path = os.path.join(output_dir, f"baseline5_{year}_{day+1}_{hour*6}_{(hour+1)*6}.nc")
            baseline_hour = xr.Dataset(
            {
            "pr": (("time", "lat", "lon"), baseline_hour) 
            },
            )
            baseline_hour.to_netcdf(out_path)
            #print(f"Saved: {out_path}")

final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape

# Baseline 6
## Using the temporal pattern of region one of the northeastern, the third quantile 50 %

In [12]:
csv_path = r"/scratch/jl14811/validation/Baselines/ne_1_24h_temporal.csv"
# Skip header lines and load 48 rows of data
skip_lines = 117
df = pd.read_csv(csv_path, skiprows=skip_lines, nrows=48)
# Extract column '0.9' and add a zero at the start
values = df['0.5'].astype(float).to_numpy()
values = np.insert(values, 0, 0.0)
# Optional: check the result
print(values.shape)
print(values)  # preview first few values
# Create slices for the even-indexed values
v_even_start = values[0:-2:2]  
v_even_end   = values[2::2]    
# Compute the differences
temp_pater_5= v_even_end - v_even_start
# Print results
print("Shape:", temp_pater_5.shape)
print("First 5 differences:", temp_pater_5[:5])  

(49,)
[  0.     1.46   2.47   3.32   4.16   5.07   6.04   7.07   8.15   9.26
  10.39  11.54  12.72  13.94  15.22  16.57  17.99  19.52  21.17  22.96
  24.92  27.07  29.44  32.06  34.97  38.19  41.72  45.56  49.69  54.08
  58.65  63.31  67.99  72.55  76.9   80.93  84.55  87.72  90.41  92.63
  94.43  95.88  97.04  98.    98.78  99.39  99.79  99.96 100.  ]
Shape: (24,)
First 5 differences: [2.47 1.69 1.88 2.11 2.24]


### Apply it to generate hourly precipitation fields for a day

In [13]:
for year in range(1981, 2012):
    input = get_data(year)
    # Output directory
    output_dir = "/scratch/jl14811/validation/Baselines/baseline6/"
    os.makedirs(output_dir, exist_ok=True)
    days = input.shape[0]
    # Create 6-hourly flat 
    for day in range(days):
        for i, hour in enumerate(range(4)):
            weight = temp_pater_5[i] / 100
            baseline_hour = np.expand_dims(input[day] * weight, axis=0)
            out_path = os.path.join(output_dir, f"baseline6_{year}_{day+1}_{hour*6}_{(hour+1)*6}.nc")
            baseline_hour = xr.Dataset(
            {
            "pr": (("time", "lat", "lon"), baseline_hour) 
            },
            )
            baseline_hour.to_netcdf(out_path)
            #print(f"Saved: {out_path}")

final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape

# Baseline 7
## Using the temporal pattern of region one of the northeastern, the fourth quantile 50 %

In [14]:
csv_path = r"/scratch/jl14811/validation/Baselines/ne_1_24h_temporal.csv"
# Skip header lines and load 48 rows of data
skip_lines = 171
df = pd.read_csv(csv_path, skiprows=skip_lines, nrows=48)
# Extract column '0.9' and add a zero at the start
values = df['0.5'].astype(float).to_numpy()
values = np.insert(values, 0, 0.0)
# Optional: check the result
print(values.shape)
print(values)  # preview first few values
# Create slices for the even-indexed values
v_even_start = values[0:-2:2]  
v_even_end   = values[2::2]    
# Compute the differences
temp_pater_6= v_even_end - v_even_start
# Print results
print("Shape:", temp_pater_6.shape)
print("First 5 differences:", temp_pater_6[:5])  

(49,)
[  0.     1.59   2.91   4.07   5.21   6.41   7.7    9.07  10.48  11.89
  13.24  14.51  15.67  16.73  17.7   18.62  19.52  20.44  21.4   22.43
  23.54  24.74  26.    27.34  28.72  30.13  31.57  33.03  34.52  36.06
  37.68  39.43  41.35  43.5   45.93  48.72  51.89  55.49  59.52  63.96
  68.75  73.79  78.93  83.96  88.63  92.71  95.97  98.33 100.  ]
Shape: (24,)
First 5 differences: [2.91 2.3  2.49 2.78 2.76]


### Apply it to generate hourly precipitation fields for a day

In [15]:
for year in range(1981, 2012):
    input = get_data(year)
    # Output directory
    output_dir = "/scratch/jl14811/validation/Baselines/baseline7/"
    os.makedirs(output_dir, exist_ok=True)
    days = input.shape[0]
    # Create 6-hourly flat 
    for day in range(days):
        for i, hour in enumerate(range(4)):
            weight = temp_pater_6[i] / 100
            baseline_hour = np.expand_dims(input[day] * weight, axis=0)
            out_path = os.path.join(output_dir, f"baseline7_{year}_{day+1}_{hour*6}_{(hour+1)*6}.nc")
            baseline_hour = xr.Dataset(
            {
            "pr": (("time", "lat", "lon"), baseline_hour) 
            },
            )
            baseline_hour.to_netcdf(out_path)
            #print(f"Saved: {out_path}")

final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape

# Baseline 8
## Using the temporal pattern of region one of the northeastern, all cases 50 %

In [16]:
csv_path = r"/scratch/jl14811/validation/Baselines/ne_1_24h_temporal.csv"
# Skip header lines and load 48 rows of data
skip_lines = 225
df = pd.read_csv(csv_path, skiprows=skip_lines, nrows=48)
# Extract column '0.9' and add a zero at the start
values = df['0.5'].astype(float).to_numpy()
values = np.insert(values, 0, 0.0)
# Optional: check the result
print(values.shape)
print(values)  # preview first few values
# Create slices for the even-indexed values
v_even_start = values[0:-2:2]  
v_even_end   = values[2::2]    
# Compute the differences
temp_pater_7= v_even_end - v_even_start
# Print results
print("Shape:", temp_pater_7.shape)
print("First 5 differences:", temp_pater_7[:5])  

(49,)
[  0.     1.63   3.31   5.     6.73   8.57  10.55  12.67  14.93  17.31
  19.78  22.31  24.89  27.49  30.12  32.79  35.49  38.24  41.05  43.92
  46.84  49.81  52.81  55.82  58.81  61.78  64.69  67.53  70.28  72.94
  75.5   77.96  80.32  82.58  84.75  86.82  88.78  90.63  92.35  93.93
  95.34  96.58  97.63  98.48  99.13  99.59  99.86  99.98 100.  ]
Shape: (24,)
First 5 differences: [3.31 3.42 3.82 4.38 4.85]


### Apply it to generate hourly precipitation fields for a day

In [ ]:
for year in range(1981, 2012):
    input = get_data(year)
    # Output directory
    output_dir = "/scratch/jl14811/validation/Baselines/baseline8/"
    os.makedirs(output_dir, exist_ok=True)
    days = input.shape[0]
    # Create 6-hourly flat 
    for day in range(days):
        for i, hour in enumerate(range(4)):
            weight = temp_pater_7[i] / 100
            baseline_hour = np.expand_dims(input[day] * weight, axis=0)
            out_path = os.path.join(output_dir, f"baseline8_{year}_{day+1}_{hour*6}_{(hour+1)*6}.nc")
            baseline_hour = xr.Dataset(
            {
            "pr": (("time", "lat", "lon"), baseline_hour) 
            },
            )
            baseline_hour.to_netcdf(out_path)
            #print(f"Saved: {out_path}")

final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape (365, 1, 21, 31)
(365, 21, 31)
final shape